In [ ]:

import os
# ---- Main folder = the repository folder, which holds Data/ (layout described in Data/README.md). ----
# Run this from the folder the file is in, or set the environment variable GUIDESTAR_MAIN_DIR to the repository path.
main_dir = os.environ.get('GUIDESTAR_MAIN_DIR', '..')



# Image alignment on LM50GS data 

Registering bits/raw tiff bits to DAPI

Code adapted from: <repo>\GS2GS_validation_ver2\LM50GS_double_GS_validation_ver6.py


In [36]:
import pandas as pd
import os
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root)) # go back to root dir
from BFT_decoding.correctionFunctions import parseDaybook
from Guidestar_colocalisation.ColocalisationPipelineFunctions_v3 import *
import yaml
import re
from skimage.transform import warp
import numpy as np
import tifffile
from scipy.ndimage import shift as ndi_shift

def txt_to_dict(yml_file):
    with open(yml_file) as file:
        user_dict= yaml.load(file, Loader=yaml.FullLoader)
    return user_dict


In [37]:
### Input Parameters ###
#mBrain
hyb_list = [1, 1, 2, 3, 4, 2, 3, 5, 1, 2, 4, 5, 6, 3, 4, 5]
type_list = ['Cy5', 'Cy7', 'Cy7', 'Cy7', 'Cy7', 'Cy5', 'Cy5', 'Cy7', 'Alexa594', 'Alexa594', 'Cy5', 'Cy5', 'Cy5', 'Alexa594', 'Alexa594', 'Alexa594']
num_bits = 16
raw_img_folder = os.path.join(main_dir, 'Data/20250728_Lm50GS_mbrain_3cT3')
save_regi_img = os.path.join(raw_img_folder, 'registered_img')
if not os.path.exists(save_regi_img):
    os.makedirs(save_regi_img)

# above combination corresponds to merfish bits
codebook_df = pd.read_csv(os.path.join(raw_img_folder,'codebook/codebook_data.tsv'), sep='\t', header=None)
triton_channel_dict = {'Cy7':0, #channel name: frame number in tif
                       'Cy5':1,
                       'Alexa594':2} #usually its the same across all images in Triton. If otherwise, need to modify the code to adapt

## chromatic correction functions ##
bft_parent = '..'
if bft_parent not in sys.path:
    sys.path.insert(0, bft_parent)

colour_mapping = {
    "Cy3": "Cy3",
    "Alexa594": "Alexa594",
    "Cy5": "Cy5",
    "Cy7": "Cy7"
}
reference_colour = "Cy5"
# 1) Correct chromatic distortion
pairs_tf_dict = {}
calibration_path = os.path.join(main_dir, 'Data/20250728_Lm50GS_mbrain_3cT3/calibrationTriton')
params_pattern1 = re.compile("[a-zA-Z]+(\d+)_[a-zA-Z]+(\d+)")
params_pattern2 = re.compile("([a-zA-Z0-9]+)to([a-zA-Z0-9]+)")

# 1) Chromatic correction
for file in os.listdir(calibration_path):
    if os.path.splitext(file)[-1] == ".csv":
        match1 = re.search(params_pattern1, file)
        match2 = re.search(params_pattern2, file)
        if match1:
            match = match1
        elif match2:
            match = match2
        daybook_fullpath = os.path.join(calibration_path, file)
        colour_pair = (match.group(1), match.group(2))
        pairs_tf_dict[colour_pair] = parseDaybook(daybook_fullpath)

colours_tf_dict = {reference_colour: None}
for pair in pairs_tf_dict:
    if pair[0] == reference_colour:
        colours_tf_dict[pair[1]] = pairs_tf_dict[pair]
    if pair[1] == reference_colour:
        colours_tf_dict[pair[0]] = pairs_tf_dict[pair].inverse

____ Matrix: ____
 [[ 1.00126148e+00  3.97436125e-05 -2.18404367e+00]
 [-3.97436125e-05  1.00126148e+00 -1.11771017e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
____ Matrix: ____
 [[ 1.00196723e+00  4.26314977e-05 -3.17444090e+00]
 [-4.26314977e-05  1.00196723e+00 -2.59335733e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
____ Matrix: ____
 [[ 9.97952527e-01  5.40682540e-07  2.15458376e+00]
 [-5.40682540e-07  9.97952527e-01  2.22519960e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [38]:
from scipy.ndimage import gaussian_filter, fourier_shift
def norm_morph(img, low_sigma=8.0):
    img = np.nan_to_num(img).astype(np.float64)
    # robust scale: ignore hot pixels and the dim floor
    lo, hi = np.percentile(img, (0, 99.9))
    d = np.clip((img - lo) / (hi - lo + 1e-9), 0, 1)
    d = gaussian_filter(d, low_sigma)
    d = d - gaussian_filter(d, 4 * low_sigma)
    return d

def estimate_shift(ref_slice, current_slice, upsample_factor):
    ref = np.nan_to_num(ref_slice).astype(np.float64)
    cur = np.nan_to_num(current_slice).astype(np.float64)

    ref_f = np.fft.fftn(ref)
    cur_f = np.fft.fftn(cur)
    shifts, fine_error, pixel_error = register_translation(
        ref_f, cur_f, upsample_factor=upsample_factor, space="fourier")

    # Score = Pearson correlation of the ALIGNED images at the returned shift.
    # Independent of register_translation's internal CCmax scaling.
    cur_aligned = np.fft.ifftn(fourier_shift(cur_f, shifts)).real
    r = (ref - ref.mean()) / (ref.std() + 1e-9)
    a = (cur_aligned - cur_aligned.mean()) / (cur_aligned.std() + 1e-9)
    ncc_peak = float(np.mean(r * a))   # ~1 identical, ~0 unrelated; in [-1, 1]

    return np.asarray(shifts, float), ncc_peak, float(np.abs(pixel_error))

def _shift_ok(shift, score, max_shift, min_score):
    mag = float(np.hypot(*shift[:2]))
    if not np.isfinite(score):
        return False, 'non-finite score'
    if mag > max_shift:
        return False, f'shift {mag:.1f}px > max_shift {max_shift}'
    if score < min_score:
        return False, f'ncc peak {score:.4g} < min_score {min_score}'
    return True, ''

def _load_dapi(root, fov):
    path = os.path.join(root, f'DAPI_1/DAPI_1_MMStack_1-Pos_{fov}.ome.tif')
    if not os.path.exists(path):
        return None
    return tifffile.imread(path).astype(np.float64)

def _crop(img, m=150): # m is the margin
    return img[m:-m, m:-m]

def _find_tiff(s10_root, fov, hyb):
    """Locate the GS TIFF for this fov/hyb under S10. Returns path or None."""
    pat = re.compile(rf'^hyb0*{int(hyb)}_\d+_MMStack_1-Pos_{fov}\.ome\.tif$')
    for root, _dirs, files in os.walk(s10_root):
        for f in sorted(files):
            if pat.match(f):
                return os.path.join(root, f)
    return None

def _log(msg, log_path=None):
    print(msg)
    if log_path is not None:
        with open(log_path, 'a') as f:
            f.write(msg + '\n')

In [39]:
def register_bit_to_dapi(
    s10_root,
    fov,
    ch_hyb,
    dapi_morph,                      # <-- pre-normalized DAPI, passed in
    triton_channel_dict,
    colours_tf_dict,
    reg_log=None,
    log_path=None,
    upsample_factor=50,
    morph_gs_low_sigma=10.0,
    max_shift_px=150.0,
    min_gs_score=0.10,
):
    """Register ONE bit (channel_hyb) of ONE FOV to a pre-normalized DAPI.
 
    Returns the shifted raw GS image (gs_regi), or None on skip/failure.
    """
    channel_name, hyb = ch_hyb.split('_')
    channel_idx = triton_channel_dict.get(channel_name)
    if channel_idx is None:
        _log(f'[skip] FOV {fov} {ch_hyb}: channel {channel_name} not in '
             f'triton_channel_dict', log_path)
        return None
 
    full_path = _find_tiff(s10_root, fov, hyb)
    if full_path is None:
        _log(f'[skip] FOV {fov} {ch_hyb}: no TIFF for hyb {hyb}', log_path)
        return None
 
    tiff_img = tifffile.imread(full_path)
    frame_axis = int(np.argmin(tiff_img.shape))
    gs_signal = np.take(tiff_img, channel_idx, axis=frame_axis)
    if colours_tf_dict.get(channel_name) is not None: # apply chromatic correction
        gs_signal = warp(gs_signal, colours_tf_dict[channel_name],
                         preserve_range=True, order=1)
 
    gs_shift, gs_score, _ = estimate_shift(
        _crop(dapi_morph),
        _crop(norm_morph(gs_signal, morph_gs_low_sigma)),
        upsample_factor,
    )
 
    ok_g, why_g = _shift_ok(gs_shift, gs_score, max_shift_px, min_gs_score)
    if reg_log is not None:
        reg_log.append(dict(fov=fov, channel_hyb=ch_hyb, step='gs->dapi',
                            shift=gs_shift, score=gs_score, ok=ok_g, reason=why_g))
    if not ok_g:
        _log(f'[skip] FOV {fov} {ch_hyb} GS->DAPI failed: {why_g}', log_path)
        return gs_signal # save the unshifted image instead of returning None, so that we can still use it for downstream analysis
 
    gs_regi = ndi_shift(gs_signal, gs_shift, order=1)
    return gs_regi

def register_all_bits_to_dapi(
    raw_img_folder,
    fovlist,
    hyb_list,
    type_list,
    triton_channel_dict,
    output_dir,
    log_path=None,
    overwrite=False,
    **reg_kwargs,
):
    """Loop over FOVs (DAPI loaded once each), register every bit, and save.
 
    Each registered image is written to
        {output_dir}/REGI_{ch_hyb}_{fov}.tiff
 
    Returns
    -------
    reg_log : list of dict
        One record per registration attempt (includes output path on success).
    """
    s10_root = os.path.join(raw_img_folder, 'S10')
    os.makedirs(output_dir, exist_ok=True)
    reg_log = []
 
    for fov_tuple in sorted(set(fovlist)):
        fov = f'{fov_tuple[0]:03d}_{fov_tuple[1]:03d}'
 
        # ---- load + normalize DAPI ONCE for this FOV ----
        dapi = _load_dapi(raw_img_folder, fov)
        if dapi is None:
            _log(f'[skip] missing DAPI for FOV {fov}', log_path)
            continue
        dapi_morph = norm_morph(dapi, reg_kwargs.get('morph_low_sigma', 8.0))
 
        # ---- register every bit against the same dapi_morph ----
        for i, (hyb, ch) in enumerate(zip(hyb_list, type_list)):
            ch_hyb = f'{ch}_{hyb}'
            out_path = os.path.join(output_dir, f'REGI_{ch_hyb}_{fov}.tiff')
 
            if os.path.exists(out_path) and not overwrite:
                print(f'FOV {fov} | bit {i+1}/{len(hyb_list)}: {ch_hyb} '
                      f'(exists, skip)')
                continue
 
            print(f'FOV {fov} | bit {i+1}/{len(hyb_list)}: {ch_hyb}')
            gs_regi = register_bit_to_dapi(
                s10_root, fov, ch_hyb, dapi_morph, triton_channel_dict,
                reg_log=reg_log, log_path=log_path, colours_tf_dict=colours_tf_dict,
                **{k: v for k, v in reg_kwargs.items() if k != 'morph_low_sigma'},
            )
            # if gs_regi is None:
            #     continue
 
            # Source TIFFs are 16-bit unsigned (sampleformat=1). ndi_shift
            # returns float64; round back to uint16 to match the original.
            # order=1 (bilinear) output is bounded by neighbors so it won't
            # overshoot, but clip is cheap insurance against edge cases.
            info = np.iinfo(np.uint16)
            gs_regi = np.clip(np.rint(gs_regi), info.min, info.max).astype(np.uint16)
            tifffile.imwrite(out_path, gs_regi)
            reg_log[-1]['out_path'] = out_path
 
    return reg_log

In [40]:
fovlist = [(i, j) for i in range(10) for j in range(10)]

reg_log = register_all_bits_to_dapi(
        raw_img_folder,
        fovlist,
        hyb_list,
        type_list,
        triton_channel_dict,
        save_regi_img,
        log_path=os.path.join(raw_img_folder, 'registration_log.txt'),
    )

FOV 000_000 | bit 1/16: Cy5_1
FOV 000_000 | bit 2/16: Cy7_1
FOV 000_000 | bit 3/16: Cy7_2
FOV 000_000 | bit 4/16: Cy7_3
FOV 000_000 | bit 5/16: Cy7_4
FOV 000_000 | bit 6/16: Cy5_2
FOV 000_000 | bit 7/16: Cy5_3
FOV 000_000 | bit 8/16: Cy7_5
FOV 000_000 | bit 9/16: Alexa594_1
FOV 000_000 | bit 10/16: Alexa594_2
FOV 000_000 | bit 11/16: Cy5_4
FOV 000_000 | bit 12/16: Cy5_5
FOV 000_000 | bit 13/16: Cy5_6
FOV 000_000 | bit 14/16: Alexa594_3
FOV 000_000 | bit 15/16: Alexa594_4
FOV 000_000 | bit 16/16: Alexa594_5


OME series: not an ome-tiff master file


FOV 000_001 | bit 1/16: Cy5_1


OME series: not an ome-tiff master file
OME series: not an ome-tiff master file


FOV 000_001 | bit 2/16: Cy7_1


OME series: not an ome-tiff master file


... (2,199 outputs left out)

OME series: not an ome-tiff master file
OME series: not an ome-tiff master file


FOV 009_009 | bit 15/16: Alexa594_4


OME series: not an ome-tiff master file


FOV 009_009 | bit 16/16: Alexa594_5


In [41]:
print('Done!')

Done!


In [28]:
# chek img type in original tiff
import tifffile, numpy as np

path = os.path.join(main_dir, 'Data/20250728_Lm50GS_mbrain_3cT3/S10/hyb02_1/hyb02_1_MMStack_1-Pos_000_001.ome.tif')

# 1) dtype from the header, no full load needed
with tifffile.TiffFile(path) as tif:
    p = tif.pages[0]
    print(p.dtype, p.bitspersample, 'sampleformat', p.sampleformat)
    # sampleformat: 1 = unsigned int, 2 = signed int, 3 = IEEE float

# 2) actual values
img = tifffile.imread(path)
print(img.dtype, img.min(), img.max())

OME series: not an ome-tiff master file


uint16 16 sampleformat 1
uint16 527 10798
